In [0]:
countries = [("USA", 10000, 20000), ("India", 1000, 1500), ("UK", 7000, 10000), ("Canada", 500, 700) ]
columns = ["Country","NumVaccinated","AvailableDoses"]

In [0]:
spark.createDataFrame(data=countries, schema=columns).write.mode("overwrite").saveAsTable("practice_workspace.default.cdc_practice_silver")

In [0]:
%sql

select * from practice_workspace.default.cdc_practice_silver

# creating gold table from silver table

In [0]:
import pyspark.sql.functions as F

spark.read.format("delta").table("practice_workspace.default.cdc_practice_silver")\
    .withColumn("VaccinationRate",F.col("NumVaccinated")/F.col("AvailableDoses"))\
    .drop("NumVaccinated","AvailableDoses")\
    .write.format("delta").mode("overwrite").saveAsTable("practice_workspace.default.cdc_practice_gold")

In [0]:
%sql

select * from practice_workspace.default.cdc_practice_gold

# Enable change data feed on silver tablem

In [0]:
%sql
ALTER TABLE practice_workspace.default.cdc_practice_silver SET TBLPROPERTIES (delta.enableChangeDataFeed = True)

# Update silver table daily

## insert new records

In [0]:
new_country = [("Australia",100,3000)]
spark.createDataFrame(data=new_country,
                      schema=columns)\
      .write.format("delta").mode("append").saveAsTable("practice_workspace.default.cdc_practice_silver")                    

In [0]:
%sql
-- update a record
UPDATE practice_workspace.default.cdc_practice_silver SET NumVaccinated = '11000' WHERE Country = 'USA'

In [0]:
%sql
-- delete a record
DELETE from practice_workspace.default.cdc_practice_silver WHERE Country = 'UK'

# Explore the change data in SQL and PySpark

In [0]:
%sql

select * from table_changes("practice_workspace.default.cdc_practice_silver",1)

In [0]:
changes_df = spark.read.format("delta").option("readChangeData", True).option("startingVersion", 2).table('practice_workspace.default.cdc_practice_silver')
display(changes_df)

# Propagate changes from silver to gold table

### creating a temp view for holding the recent data

In [0]:
%sql
create or replace temporary view cdc_practice_silver_changes as
select * from (select *,rank() over (partition by Country order by _commit_version desc) as rank
from table_changes("practice_workspace.default.cdc_practice_silver",2) where _change_type != 'update_preimage')
where rank = 1
    

#  Merge the changes to gold

In [0]:
%sql

Merge  into practice_workspace.default.cdc_practice_gold as t
using cdc_practice_silver_changes as s
on t.Country = s.Country
when matched and s._change_type = 'update_postimage' then update set VaccinationRate = s.NumVaccinated/s.AvailableDoses
when not matched then insert (Country,VaccinationRate) values (s.Country,s.NumVaccinated/s.AvailableDoses)

In [0]:
%sql

select * from practice_workspace.default.cdc_practice_gold